# Lezione 11 · L'economia del gesto: inferenza e ottimizzazioni

Il venditore del molo e il viaggio dei pezzi: il viaggio e la busta in numeri, i quattro pesi scritti con pochi bit, il sapere oscuro, i reparti, e il conto in euro.

Ogni numero calcolato qui viene confrontato con quello scritto nel libro: un segno ✓ vuol dire che coincidono.

In [1]:
import math
import numpy as np

def controlla(nome, valore, atteso, tolleranza=0.01):
    """Confronta un valore calcolato con quello scritto nel libro."""
    ok = abs(valore - atteso) <= tolleranza
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    print(f"{segno}  {nome}: calcolato {valore:.4g}, nel libro {atteso:g}")
    assert ok, nome

def conta(nome, valore, atteso):
    """Come controlla, per i numeri interi: devono coincidere esattamente."""
    ok = valore == atteso
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    intero = lambda x: f"{x:,.0f}".replace(",", ".")
    print(f"{segno}  {nome}: calcolato {intero(valore)}, nel libro {intero(atteso)}")
    assert ok, nome

## Il viaggio in numeri

*Nel racconto: i 140 GB da mettere in moto per ogni token; gli addetti che «su un'ora, lavorano meno di un minuto».*

La rete piccola ha otto pesi. Un modello da 70 miliardi, scritto a 16 bit, pesa 140 GB, e per ogni token li mette in moto tutti: quanto in fretta, lo decide la banda.

In [2]:
conta("pesi della rete piccola, 3 → 2 → 1", 3 * 2 + 2 * 1, 8)

pesi = 70e9 * 2                      # 70 miliardi di parametri a 16 bit, in byte
controlla("pesi (GB)", pesi / 1e9, 140, 0)

banda_h100 = 3300e9                  # byte al secondo, acceleratore di punta del 2023
banda_b200 = 8000e9                  # acceleratore del 2025
controlla("secondi per token", pesi / banda_h100, 0.04, 0.003)
controlla("token al secondo, «una ventina»", banda_h100 / pesi, 20, 4)
controlla("token al secondo a 8.000 miliardi di byte, «quasi sessanta»", banda_b200 / pesi, 57, 3)

✓  pesi della rete piccola, 3 → 2 → 1: calcolato 8, nel libro 8
✓  pesi (GB): calcolato 140, nel libro 140
✓  secondi per token: calcolato 0.04242, nel libro 0.04
✓  token al secondo, «una ventina»: calcolato 23.57, nel libro 20
✓  token al secondo a 8.000 miliardi di byte, «quasi sessanta»: calcolato 57.14, nel libro 57


Per ogni token il modello fa circa due operazioni per parametro. Il chip, sulla carta, ne saprebbe fare circa trecento volte tanto: per questo gli addetti aspettano.

In [3]:
operazioni_per_token = 2 * 70e9
conta("operazioni per token (miliardi)", operazioni_per_token / 1e9, 140)

al_secondo = operazioni_per_token * 20          # a venti token al secondo
print(f"a venti token al secondo: {al_secondo / 1e9:,.0f} miliardi di operazioni al secondo".replace(",", "."))
assert al_secondo < 3000e9                      # «meno di tremila miliardi»

secondi_di_lavoro = 3600 / 300                  # il chip saprebbe farne circa trecento volte tanto
print(f"su un'ora, gli addetti lavorano {secondi_di_lavoro:.0f} secondi")
assert secondi_di_lavoro < 60                   # «meno di un minuto»

✓  operazioni per token (miliardi): calcolato 140, nel libro 140
a venti token al secondo: 2.800 miliardi di operazioni al secondo
su un'ora, gli addetti lavorano 12 secondi


La lettura è un altro conto: una pagina da 10.000 token chiede circa 2 × 70 miliardi × 10.000 operazioni, e qui il venditore lavora davvero.

In [4]:
lettura = 2 * 70e9 * 10_000
conta("operazioni per leggere 10.000 token (miliardi)", lettura / 1e9, 1_400_000)

sulla_carta = 300 * al_secondo                  # operazioni al secondo dell'acceleratore grande, sulla carta
print(f"acceleratore grande, sulla carta: {lettura / sulla_carta:.1f} secondi; nel libro «due o tre secondi», perché nella pratica il chip fa un po' meno")

# un buon computer di casa: da dieci a ventimila miliardi di operazioni al secondo
controlla("computer di casa, a ventimila miliardi al secondo (minuti)", lettura / 20_000e9 / 60, 1, 0.2)
controlla("computer di casa, a diecimila miliardi al secondo (minuti)", lettura / 10_000e9 / 60, 2.5, 0.2)

✓  operazioni per leggere 10.000 token (miliardi): calcolato 1.400.000, nel libro 1.400.000
acceleratore grande, sulla carta: 1.7 secondi; nel libro «due o tre secondi», perché nella pratica il chip fa un po' meno
✓  computer di casa, a ventimila miliardi al secondo (minuti): calcolato 1.167, nel libro 1
✓  computer di casa, a diecimila miliardi al secondo (minuti): calcolato 2.333, nel libro 2.5


## La busta in numeri

*Nel racconto: la busta dei documenti, che ingrassa a ogni viaggio; il mezzo milione di controlli contro mille; «ogni parola aggiunge alla busta circa un terzo di megabyte».*

La busta è la KV cache. Si conta così: byte per token = 2 × strati × teste KV × dimensione della testa × byte per numero. Il 2 c'è perché ogni token lascia due cose, la key e il value.

In [5]:
def byte_per_token(strati, teste_kv, dim_testa, byte=2):
    return 2 * strati * teste_kv * dim_testa * byte

bpt = byte_per_token(80, 8, 128)     # 80 strati, 8 teste KV di dimensione 128, a 16 bit
conta("byte per token", bpt, 327_680)
controlla("megabyte per token, «circa un terzo»", bpt / 1e6, 1 / 3, 0.01)

busta = bpt * 128_000 / 1e9
controlla("busta a 128.000 token (GB)", busta, 42, 0.1)
controlla("in rapporto ai pesi, «quasi un terzo»", busta / 140, 1 / 3, 0.04)

# senza la GQA, le 64 teste di domanda avrebbero ciascuna la sua key e il suo value
conta("teste di domanda per ogni testa KV", 64 // 8, 8)
controlla("busta senza GQA (GB)", byte_per_token(80, 64, 128) * 128_000 / 1e9, 335, 1)

✓  byte per token: calcolato 327.680, nel libro 327.680
✓  megabyte per token, «circa un terzo»: calcolato 0.3277, nel libro 0.333333
✓  busta a 128.000 token (GB): calcolato 41.94, nel libro 42
✓  in rapporto ai pesi, «quasi un terzo»: calcolato 0.2996, nel libro 0.333333
✓  teste di domanda per ogni testa KV: calcolato 8, nel libro 8
✓  busta senza GQA (GB): calcolato 335.5, nel libro 335


Il mezzo milione del racconto: senza la busta, per scrivere n token si ripassano 1 + 2 + … + (n − 1) = n(n − 1) / 2 token; con la busta, n.

In [6]:
n = 1000
senza_busta = sum(range(1, n))       # uno più due più tre, fino a novecentonovantanove
assert senza_busta == n * (n - 1) // 2
conta("controlli senza la busta", senza_busta, 499_500)
conta("controlli con la busta", n, 1000)

✓  controlli senza la busta: calcolato 499.500, nel libro 499.500
✓  controlli con la busta: calcolato 1.000, nel libro 1.000


## La tabella a pezzi

*Nel racconto: il venditore al tavolino, dieci prodotti contro dieci; il taccuino con due cose sole, l'abbinamento migliore finora e quanti vanno bene, cioè il massimo e la somma.*

La tabella cresce con il quadrato della lista.

In [7]:
conta("caselle con cento prodotti", 100 * 100, 10_000)
conta("caselle con mille prodotti", 1000 * 1000, 1_000_000)
conta("caselle con una lista di 100.000 token (miliardi)", 100_000 ** 2 / 1e9, 10)

✓  caselle con cento prodotti: calcolato 10.000, nel libro 10.000
✓  caselle con mille prodotti: calcolato 1.000.000, nel libro 1.000.000
✓  caselle con una lista di 100.000 token (miliardi): calcolato 10, nel libro 10


Il taccuino in codice: una riga della tabella, cento punteggi, letta dieci alla volta. Si tengono da parte solo il massimo e la somma parziale, e alla fine la softmax è la stessa di chi ha steso la riga intera. I punteggi sono tirati a caso: il conto vale per qualunque riga.

In [8]:
rng = np.random.default_rng(0)
punteggi = rng.normal(size=100)      # una riga della tabella: un prodotto contro gli altri cento
valori = rng.normal(size=100)        # ciò che ogni prodotto consegna

# tutta la riga stesa sul tabellone
pesi_softmax = np.exp(punteggi - punteggi.max())
intero = (pesi_softmax * valori).sum() / pesi_softmax.sum()

# dieci alla volta, sul tavolino
massimo, somma, risultato = -np.inf, 0.0, 0.0
for i in range(0, 100, 10):
    s, v = punteggi[i:i + 10], valori[i:i + 10]
    nuovo_massimo = max(massimo, s.max())
    correzione = math.exp(massimo - nuovo_massimo)       # si corregge ciò che era già sul taccuino
    somma = somma * correzione + np.exp(s - nuovo_massimo).sum()
    risultato = risultato * correzione + (np.exp(s - nuovo_massimo) * v).sum()
    massimo = nuovo_massimo

print(f"riga intera: {intero:.6f}   a riquadri: {risultato / somma:.6f}")
assert np.isclose(intero, risultato / somma)
print("✓  il risultato è identico")

riga intera: -0.020093   a riquadri: -0.020093
✓  il risultato è identico


## Il garzone e il magazziniere

*Nel racconto: i cinque pezzi proposti dal garzone, approvati fino al primo errore.*

Il garzone propone cinque pezzi; il magazziniere ne approva tre, corregge il quarto, e il quinto torna sullo scaffale.

In [9]:
proposti = 5
approvati = 3
token_per_viaggio = approvati + 1    # i tre approvati, più il pezzo giusto messo al posto del quarto
print(f"{proposti} pezzi proposti, {approvati} approvati, 1 corretto: {token_per_viaggio} token con un solo viaggio dei pesi")

5 pezzi proposti, 3 approvati, 1 corretto: 4 token con un solo viaggio dei pesi


Ogni token proposto x si accetta con probabilità min(1, p(x) / q(x)); al primo rifiuto se ne pesca uno nuovo da una distribuzione proporzionale a max(0, p − q). Il risultato ha esattamente la distribuzione p del magazziniere. Qui p e q sono tirate a caso, su cinque token possibili: il conto vale per qualunque garzone.

In [10]:
p = rng.dirichlet(np.ones(5))                    # il magazziniere, il modello grande
q = rng.dirichlet(np.ones(5))                    # il garzone, il modello piccolo

accettato = q * np.minimum(1, p / q)             # il garzone propone x, e il magazziniere lo accetta
rifiuto = 1 - accettato.sum()                    # probabilità che la proposta venga rifiutata
corretta = np.maximum(0, p - q)
corretta = corretta / corretta.sum()             # da dove si pesca dopo un rifiuto
finale = accettato + rifiuto * corretta

print("garzone      q:", np.round(q, 3))
print("magazziniere p:", np.round(p, 3))
print("risultato     :", np.round(finale, 3))
assert np.allclose(finale, p)
print("✓  il garzone cambia la velocità, non il testo")

garzone      q: [0.259 0.216 0.16  0.272 0.093]
magazziniere p: [0.104 0.091 0.356 0.399 0.051]
risultato     : [0.104 0.091 0.356 0.399 0.051]
✓  il garzone cambia la velocità, non il testo


## I numeri più corti

*Nel racconto: i quattro pesi a 8 e a 4 bit, il gigante, la carta a quindici colori, quella di dettaglio per la spiaggia e quella d'insieme per la fossa.*

Si decide quante fasce avere, e ogni peso finisce nella fascia più vicina. Il più grande in valore assoluto decide la scala.

In [11]:
def quantizza(pesi, bit):
    livelli = 2 ** (bit - 1) - 1          # 127 a 8 bit, 7 a 4 bit
    scala = np.max(np.abs(pesi)) / livelli
    interi = np.round(pesi / scala)
    return scala, interi, interi * scala

conta("fasce a 8 bit, da −127 a 127", len(range(-127, 128)), 255)
conta("fasce a 4 bit, da −7 a 7", len(range(-7, 8)), 15)

pesi = np.array([0.12, -0.48, 0.91, -0.07])
scala, interi, ricostruiti = quantizza(pesi, 8)
controlla("scala a 8 bit", scala, 0.0072, 0.0001)
for i, att in zip(interi, [17, -67, 127, -10]):
    conta("intero a 8 bit", i, att)
for r, att in zip(ricostruiti, [0.122, -0.480, 0.910, -0.072]):
    controlla("rimoltiplicato per la scala", r, att, 0.001)

✓  fasce a 8 bit, da −127 a 127: calcolato 255, nel libro 255
✓  fasce a 4 bit, da −7 a 7: calcolato 15, nel libro 15
✓  scala a 8 bit: calcolato 0.007165, nel libro 0.0072
✓  intero a 8 bit: calcolato 17, nel libro 17
✓  intero a 8 bit: calcolato -67, nel libro -67
✓  intero a 8 bit: calcolato 127, nel libro 127
✓  intero a 8 bit: calcolato -10, nel libro -10
✓  rimoltiplicato per la scala: calcolato 0.1218, nel libro 0.122
✓  rimoltiplicato per la scala: calcolato -0.4801, nel libro -0.48
✓  rimoltiplicato per la scala: calcolato 0.91, nel libro 0.91
✓  rimoltiplicato per la scala: calcolato -0.07165, nel libro -0.072


A 4 bit gli interi vanno da −7 a 7.

In [12]:
scala4, _, r4 = quantizza(pesi, 4)
controlla("scala a 4 bit", scala4, 0.13, 0.001)
for r, att in zip(r4, [0.13, -0.52, 0.91, -0.13]):
    controlla("a 4 bit", r, att, 0.005)
print(f"il −0,07 è diventato {r4[3] / pesi[3]:.2f} volte tanto: quasi il doppio")

✓  scala a 4 bit: calcolato 0.13, nel libro 0.13
✓  a 4 bit: calcolato 0.13, nel libro 0.13
✓  a 4 bit: calcolato -0.52, nel libro -0.52
✓  a 4 bit: calcolato 0.91, nel libro 0.91
✓  a 4 bit: calcolato -0.13, nel libro -0.13
il −0,07 è diventato 1.86 volte tanto: quasi il doppio


Ora il quinto peso: 9,0. Un peso solo, e cancella tutti gli altri.

In [13]:
scala_g, _, r4g = quantizza(np.append(pesi, 9.0), 4)
controlla("scala con il gigante, 9 / 7", scala_g, 1.3, 0.02)
print("con il gigante da 9,0:", np.round(r4g, 2) + 0.0)
conta("pesi su quattro caduti sullo zero", int((r4g[:4] == 0).sum()), 3)
controlla("lo 0,91 diventa", r4g[2], 1.29, 0.01)

✓  scala con il gigante, 9 / 7: calcolato 1.286, nel libro 1.3
con il gigante da 9,0: [0.   0.   1.29 0.   9.  ]
✓  pesi su quattro caduti sullo zero: calcolato 3, nel libro 3
✓  lo 0,91 diventa: calcolato 1.286, nel libro 1.29


La baia: una fossa da novanta metri e una carta con quindici colori.

In [14]:
fascia = 90 / 15
controlla("larghezza di ogni fascia (metri)", fascia, 6, 0)
for fondale, atteso in [(0.5, 0), (2, 0), (4, 6)]:
    controlla(f"un fondale di {fondale} metri sulla carta diventa", round(fondale / fascia) * fascia, atteso, 0)

✓  larghezza di ogni fascia (metri): calcolato 6, nel libro 6
✓  un fondale di 0.5 metri sulla carta diventa: calcolato 0, nel libro 0
✓  un fondale di 2 metri sulla carta diventa: calcolato 0, nel libro 0
✓  un fondale di 4 metri sulla carta diventa: calcolato 6, nel libro 6


Quanto pesano i mezzi: 70 miliardi di parametri a 16, a 8 e a 4 bit.

In [15]:
for bit, atteso in [(16, 140), (8, 70), (4, 35)]:
    controlla(f"70 miliardi a {bit} bit (GB)", 70e9 * bit / 8 / 1e9, atteso, 0)
controlla("da 16 a 4 bit, mezzi quante volte più leggeri", 140 / 35, 4, 0)

✓  70 miliardi a 16 bit (GB): calcolato 140, nel libro 140
✓  70 miliardi a 8 bit (GB): calcolato 70, nel libro 70
✓  70 miliardi a 4 bit (GB): calcolato 35, nel libro 35
✓  da 16 a 4 bit, mezzi quante volte più leggeri: calcolato 4, nel libro 4


## Il sapere oscuro

*Nel racconto: «porto» 90%, «darsena» 8%, «ritardo» 2%.*

Le probabilità del grande si ammorbidiscono con una temperatura T: ogni punteggio si divide per T prima della softmax.

In [16]:
p = np.array([0.90, 0.08, 0.02])                 # «porto», «darsena», «ritardo»
logaritmi = np.log(p / p[0])                     # contati dal primo
for l, att in zip(logaritmi, [0, -2.42, -3.81]):
    controlla("logaritmo", l, att, 0.005)

T = 2
for l, att in zip(logaritmi / T, [0, -1.21, -1.90]):
    controlla("con T = 2", l, att, 0.005)

morbide = np.exp(logaritmi / T) / np.exp(logaritmi / T).sum()
for parola, m, att in zip(["porto", "darsena", "ritardo"], morbide, [0.69, 0.21, 0.10]):
    controlla(f"probabilità di «{parola}» con T = 2", m, att, 0.005)

✓  logaritmo: calcolato 0, nel libro 0
✓  logaritmo: calcolato -2.42, nel libro -2.42
✓  logaritmo: calcolato -3.807, nel libro -3.81
✓  con T = 2: calcolato 0, nel libro 0
✓  con T = 2: calcolato -1.21, nel libro -1.21
✓  con T = 2: calcolato -1.903, nel libro -1.9
✓  probabilità di «porto» con T = 2: calcolato 0.691, nel libro 0.69
✓  probabilità di «darsena» con T = 2: calcolato 0.206, nel libro 0.21
✓  probabilità di «ritardo» con T = 2: calcolato 0.103, nel libro 0.1


## I reparti

*Nel racconto: «quasi settecento miliardi di parametri, meno di quaranta per token».*

Il modello del racconto ha 671 miliardi di parametri, 37 attivi per token. Risparmia calcolo, non memoria: in memoria stanno tutti.

In [17]:
totali, attivi = 671, 37
assert totali < 700 and attivi < 40
print(f"al lavoro per ogni token: {attivi} miliardi su {totali}, il {attivi / totali:.1%}")
print(f"in memoria: tutti i {totali} miliardi")

al lavoro per ogni token: 37 miliardi su 671, il 5.5%
in memoria: tutti i 671 miliardi


## Il conto in euro

*Nel racconto: le due righe dello scontrino del venditore; 3 e 15 euro per milione di token, prezzi tondi di esempio; i moltiplicatori.*

La scuola di vela: mille domande al giorno da 3.000 token, risposte da 500.

In [18]:
prezzo_letto, prezzo_scritto = 3, 15             # euro per milione di token
lettura = 1000 * 3000 / 1e6 * prezzo_letto
scrittura = 1000 * 500 / 1e6 * prezzo_scritto
controlla("lettura al giorno (€)", lettura, 9, 0)
controlla("scrittura al giorno (€)", scrittura, 7.5, 0)
controlla("totale al giorno (€)", lettura + scrittura, 16.5, 0)
controlla("al mese, «circa 500» (€)", 30 * (lettura + scrittura), 500, 10)

✓  lettura al giorno (€): calcolato 9, nel libro 9
✓  scrittura al giorno (€): calcolato 7.5, nel libro 7.5
✓  totale al giorno (€): calcolato 16.5, nel libro 16.5
✓  al mese, «circa 500» (€): calcolato 495, nel libro 500


I moltiplicatori. L'agente: dieci passaggi su una lista che cresce da 3.000 a 8.000 token.

In [19]:
liste = np.linspace(3000, 8000, 10)              # la lista letta a ogni passaggio
conta("token letti per passaggio, in media", liste.mean(), 5500)
conta("token letti per domanda", liste.sum(), 55_000)
controlla("quante volte i 3.000 di partenza", liste.sum() / 3000, 18, 0.5)

# anche la scrittura si moltiplica per i passaggi
agente = 1000 * liste.sum() / 1e6 * prezzo_letto + 1000 * 10 * 500 / 1e6 * prezzo_scritto
volte = agente / (lettura + scrittura)
print(f"conto dell'agente: {agente:.0f} € al giorno, {volte:.1f} volte il conto di partenza")
assert 10 <= volte <= 30                         # «per dieci o per trenta»

controlla("venti documenti invece di cinque: la lettura si moltiplica per", 20 / 5, 4, 0)

✓  token letti per passaggio, in media: calcolato 5.500, nel libro 5.500
✓  token letti per domanda: calcolato 55.000, nel libro 55.000
✓  quante volte i 3.000 di partenza: calcolato 18.33, nel libro 18
conto dell'agente: 240 € al giorno, 14.5 volte il conto di partenza
✓  venti documenti invece di cinque: la lettura si moltiplica per: calcolato 4, nel libro 4


## L'energia di una domanda

*Nel racconto: la diceria delle dieci ricerche sul web; «quanto pochi secondi di televisione accesa».*

In [20]:
controlla("una domanda del 2023 in ricerche sul web", 3 / 0.3, 10, 0)

wattora = 0.24                                   # la richiesta di testo mediana dichiarata nel 2025
televisore = 100                                 # watt: un televisore acceso, per fare il conto
secondi = wattora * 3600 / televisore
print(f"{wattora} wattora sono {secondi:.1f} secondi di un televisore da {televisore} watt")
assert secondi < 9                               # «meno di nove secondi di televisione»

✓  una domanda del 2023 in ricerche sul web: calcolato 10, nel libro 10
0.24 wattora sono 8.6 secondi di un televisore da 100 watt


## Kata per casa

1. Quanta memoria serve per i pesi di un modello da 70 miliardi di parametri a 16, a 8 e a 4 bit?
2. Un MoE ha 8 esperti e ne attiva 2 per token. Che frazione del calcolo degli esperti usa per ogni token?

In [21]:
# Esercizio 1. Memoria = parametri × byte per parametro
for formato, byte, atteso in [("fp16", 2, 140), ("int8", 1, 70), ("int4", 0.5, 35)]:
    controlla(f"70 miliardi in {formato} (GB)", 70e9 * byte / 1e9, atteso, 0)

# Esercizio 2. Ma tutti e 8 devono stare in memoria
controlla("frazione del calcolo degli esperti (%)", 2 / 8 * 100, 25, 0)

✓  70 miliardi in fp16 (GB): calcolato 140, nel libro 140
✓  70 miliardi in int8 (GB): calcolato 70, nel libro 70
✓  70 miliardi in int4 (GB): calcolato 35, nel libro 35
✓  frazione del calcolo degli esperti (%): calcolato 25, nel libro 25
